# Nettoyage — leboncoin-private (pilier Prix)

Applique et **justifie** le nettoyage du dataset validé au Gate EDA
(`01_eda_inspection.ipynb`). Ce carnet ne modélise rien : il produit le jeu de travail et la
trace qui rend chaque suppression défendable.

Toute la logique vit dans `ml/src/leboncoin.py` — le carnet ne fait que l'appeler et rendre
compte. Le fichier `raw/` n'est jamais modifié.

**Trois arbitrages actés en amont :**

1. **Pas de filtre « sans description »** : `body` est vide sur 20 915 / 20 915 annonces, la
   description n'existe pas sur le listing leboncoin. Le re-scraping des pages annonce est un
   chantier séparé.
2. **Borne haute du prix paramétrable** : `max_price` vaut 50 000 € par défaut — la décision
   **ADR 0002** en vigueur. La comparaison 50 k / 100 k se fera à la baseline, sur l'erreur
   mesurée, pas ici par décret.
3. **Énergies : tout gardé, micro-classes regroupées.** Le motif de l'**ADR 0001** (batterie en
   location → prix non comparable) reposait sur une colonne du dataset 2023 absente ici :
   le recopier serait reprendre une conclusion sans sa preuve.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, "../../src")
from leboncoin import (COLONNES_ECARTEES, clean_leboncoin, colonnes_modele,  # noqa: E402
                       resume_journal)

RAW = Path("../../data/leboncoin-private/raw/annonces.parquet")
PREMIUM = Path("../../references/premium_brand.csv")
OUT = Path("../../data/leboncoin-private/processed/annonces_clean.parquet")

brut = pd.read_parquet(RAW)
N_DEPART = len(brut)
print(f"Depart : {N_DEPART} annonces")

Depart : 20915 annonces


## 1. Journal de nettoyage

Chaque règle, son coût en lignes, et le total qui doit boucler. Une perte non chiffrée est une
perte non défendable — c'est l'attendu de transparence du bloc BC04.

In [2]:
propre, journal = clean_leboncoin(RAW, PREMIUM)
print(resume_journal(journal, N_DEPART))

regle                                                       perdues  restantes
------------------------------------------------------------------------------
prix_eur >= 500 EUR                                             340      20575
prix_eur <= 50,000 EUR (ADR 0002)                               147      20428
annee >= 1980                                                   475      19953
quasi-doublons vehicule (marque+modele+annee+km+prix)            71      19882
kilometrage >= 999999 -> NaN (1 valeurs, aucune ligne jetee)        0      19882
mise en circulation aberrante -> repli sur annee (7 valeurs)        0      19882
------------------------------------------------------------------------------
TOTAL                                                          1033      19882
depart 20915 - 1033 perdues = 19882


### Ce que chaque règle retire, et pourquoi

| Règle | Motif |
|---|---|
| `prix_eur >= 500 €` | En dessous : prix d'appel, épaves, ventes de pièces. Pas le marché du produit. |
| `prix_eur <= 50 000 €` | Périmètre produit, **ADR 0002**. Paramétrable (`max_price`). |
| `annee >= 1980` | Véhicules de collection : cote non comparable au marché d'occasion grand public. Miroir de `features.clean_cars`. |
| Quasi-doublons véhicule | Même voiture republiée sous deux `ad_id` → sinon fuite entre entraînement et test. |
| `kilometrage >= 999 999` | Champ rempli au maximum, pas un kilométrage. On neutralise **la valeur**, pas la ligne. |
| Mise en circulation aberrante | Quelques vendeurs saisissent `12/0994` pour 1994. Format valide, année absurde → repli sur `annee`, qui est correcte. Sans ce garde-fou, l'âge partait à plus de 1 000 ans. |

## 2. Avant / après

In [3]:
comparaison = pd.DataFrame({
    "brut": brut[["prix_eur", "kilometrage", "annee"]].describe().round(0).stack(),
    "propre": propre[["prix_eur", "kilometrage", "annee"]].describe().round(0).stack(),
}).unstack(0)
print(comparaison.to_string())
print()
print(f"Lignes  : {N_DEPART} -> {len(propre)}  ({100 * len(propre) / N_DEPART:.1f} % conserve)")
print(f"Colonnes: {brut.shape[1]} -> {propre.shape[1]}")

                brut                                                                      propre                                                                   
               count      mean      std     min       25%       50%       75%       max    count      mean      std     min       25%       50%       75%       max
prix_eur     20915.0    7799.0  12616.0     1.0    1800.0    4350.0    9990.0  999999.0  19882.0    7269.0   7816.0   500.0    1900.0    4400.0    9900.0   50000.0
kilometrage  20915.0  182438.0  90229.0     1.0  118000.0  180000.0  240000.0  999999.0  19881.0  184672.0  87252.0     1.0  122000.0  181600.0  240000.0  933333.0
annee        20915.0    2009.0     10.0  1960.0    2005.0    2010.0    2015.0    2026.0  19882.0    2010.0      8.0  1980.0    2005.0    2010.0    2015.0    2026.0

Lignes  : 20915 -> 19882  (95.1 % conserve)
Colonnes: 24 -> 36


## 3. Features produites

Taux de remplissage après nettoyage. Deux colonnes restent partiellement vides et c'est assumé :
`critair` (vignette non renseignée par le vendeur) et `ct_valide_jusqu_a` (contrôle technique).
Un modèle à base d'arbres gère nativement le manquant — on ne les impute pas ici.

In [4]:
features = colonnes_modele(propre)
remplissage = (100 * propre[features].notna().mean()).round(1).sort_values()
print(f"{len(features)} features destinees au modele\n")
print(f"{'feature':<24}{'% rempli':>10}  dtype")
for nom, pct in remplissage.items():
    print(f"{nom:<24}{pct:>9.1f}%  {propre[nom].dtype}")

19 features destinees au modele

feature                   % rempli  dtype
critair                      33.1%  Int64
ct_valide_jusqu_a            53.7%  Int64
puissance_din                97.9%  float64
puissance_fisc               98.4%  float64
couleur                      99.6%  str
portes                       99.6%  Int64
places                       99.6%  Int64
age                         100.0%  float64
region                      100.0%  str
modele                      100.0%  str
marque                      100.0%  str
energie_grp                 100.0%  str
boite_auto                  100.0%  int8
niveau                      100.0%  float64
palier                      100.0%  str
kilometrage                 100.0%  Int64
annee                       100.0%  Int64
etat_niveau                 100.0%  int8
etat                        100.0%  str


In [5]:
print("Cardinalite des categorielles :")
for col in ["energie_grp", "boite_auto", "palier", "etat", "marque", "modele", "couleur", "region"]:
    print(f"  {col:<14}{propre[col].nunique():>6} modalites")
print()
print("energie_grp (micro-classes GPL / Autre / GNV fusionnees) :")
print(propre["energie_grp"].value_counts().to_string())
print()
print("Echelle d'etat (etat_niveau, ordonnee du pire au meilleur) :")
print(propre.groupby("etat_niveau", observed=True)
      .agg(etat=("etat", "first"), n=("prix_eur", "size"), prix_median=("prix_eur", "median"))
      .to_string())

Cardinalite des categorielles :
  energie_grp        6 modalites
  boite_auto         2 modalites
  palier             3 modalites
  etat               8 modalites
  marque            80 modalites
  modele           804 modalites
  couleur           17 modalites
  region            26 modalites

energie_grp (micro-classes GPL / Autre / GNV fusionnees) :
energie_grp
Diesel                  11339
Essence                  7736
Hybride                   381
Électrique                243
Autre                     107
Hybride Rechargeable       76

Echelle d'etat (etat_niveau, ordonnee du pire au meilleur) :
                               etat     n  prix_median
etat_niveau                                           
0                      not_drivable  1006       1300.0
1                           damaged   681       1500.0
2              major_repairs_needed  3181       1500.0
3              minor_repairs_needed  3248       2500.0
4              normal_wear_and_tear  3305       4500.0
5    

## 4. Colonnes écartées — et pourquoi

Le point le plus important de ce carnet. Trois de ces colonnes auraient **amélioré les métriques
de validation tout en cassant le modèle en production** : c'est la définition d'une fuite.

In [6]:
for col, motif in COLONNES_ECARTEES.items():
    print(f"--- {col} ---")
    print(f"    {motif}")
    print()
print("Note : ces colonnes ne sont pas detruites. Le Parquet raw/ n'est jamais modifie,")
print("       une jointure sur ad_id les recupere quand l'evaluation en aura besoin —")
print("       car_price_min/max servira de comparateur externe a notre fourchette.")

--- car_price_min ---
    estimation de prix de leboncoin (45 %) — le modèle recopierait un estimateur tiers au lieu du marché, et s'effondrerait sur les 55 % qui ne l'ont pas. Réservée comme comparateur externe à l'évaluation.

--- car_price_max ---
    idem car_price_min.

--- old_price ---
    prix précédent de la même annonce (9 %) — ancre quasi circulaire pour la cible. Réservé au pilier Date (baisse de prix ↔ délai de vente).

--- is_import ---
    'false' sur 20 915 / 20 915 — variance nulle, aucune information.

--- body ---
    vide sur 20 915 / 20 915 — la description n'est pas rendue sur le listing leboncoin.

Note : ces colonnes ne sont pas detruites. Le Parquet raw/ n'est jamais modifie,
       une jointure sur ad_id les recupere quand l'evaluation en aura besoin —
       car_price_min/max servira de comparateur externe a notre fourchette.


## 5. Sensibilité au périmètre de prix

`max_price` est un **paramètre**, pas une décision figée. On mesure ici ce que chaque périmètre
coûte en volume ; ce qu'il coûte en **erreur** se mesurera à la baseline, en miroir de la méthode
qui a produit l'ADR 0002. Aucun verdict dans ce carnet.

In [7]:
lignes = []
for seuil in (50_000, 100_000, None):
    sub, _ = clean_leboncoin(RAW, PREMIUM, max_price=seuil)
    lignes.append({
        "max_price": "aucun" if seuil is None else f"{seuil:,}",
        "lignes": len(sub),
        "% du dataset": round(100 * len(sub) / N_DEPART, 1),
        "prix median": round(sub["prix_eur"].median()),
        "prix p99": round(sub["prix_eur"].quantile(0.99)),
        "prix max": round(sub["prix_eur"].max()),
    })
print(pd.DataFrame(lignes).to_string(index=False))
print()
print("-> l'ecart entre 50k et 100k est de 118 annonces sur ~20 000 : sur CE dataset le seuil")
print("   haut ne mord presque plus (marche leboncoin bien plus bas, mediane ~4 350 EUR).")
print("   Le vrai arbitrage se joue en bas de distribution, pas en haut.")

max_price  lignes  % du dataset  prix median  prix p99  prix max
   50,000   19882          95.1         4400     37000     50000
  100,000   19992          95.6         4490     43000     99990
    aucun   20017          95.7         4500     44998    999999

-> l'ecart entre 50k et 100k est de 118 annonces sur ~20 000 : sur CE dataset le seuil
   haut ne mord presque plus (marche leboncoin bien plus bas, mediane ~4 350 EUR).
   Le vrai arbitrage se joue en bas de distribution, pas en haut.


## 6. Écriture du jeu nettoyé

In [8]:
OUT.parent.mkdir(parents=True, exist_ok=True)
propre.to_parquet(OUT, index=False, compression="zstd")

relu = pd.read_parquet(OUT)
print(f"Ecrit : {OUT}  ({OUT.stat().st_size / 1e6:.1f} Mo)")
print(f"Relecture : {relu.shape[0]} lignes x {relu.shape[1]} colonnes")

fuites = [c for c in COLONNES_ECARTEES if c in relu.columns]
print(f"Colonnes ecartees presentes dans le fichier : {fuites or 'aucune'}")
objets = [c for c in features if relu[c].dtype == "object"]
print(f"Features en dtype object (non typees) : {objets or 'aucune'}")

Ecrit : ../../data/leboncoin-private/processed/annonces_clean.parquet  (0.8 Mo)
Relecture : 19882 lignes x 36 colonnes
Colonnes ecartees presentes dans le fichier : aucune
Features en dtype object (non typees) : aucune


## Récapitulatif

Le jeu de travail est prêt. **Rien n'est décidé** : le verdict du candidat `leboncoin-private`
pour le pilier Prix, et le périmètre de prix définitif, s'écriront en ADR **après** la baseline —
sur des chiffres d'erreur, pas sur des intuitions.

Prochaine étape : `03_baseline_regression.ipynb`, en miroir des baselines des candidats précédents.